# Learning 02: Sentiment Analysis with GLiClass

GLiClass excels at sentiment analysis: the zero-shot approach means you can define exactly
the sentiment taxonomy relevant to your domain without any fine-tuning.

## Sentiment taxonomies

| Use case | Labels |
|---|---|
| Basic polarity | `["positive", "negative"]` |
| With neutral | `["positive", "neutral", "negative"]` |
| Fine-grained | `["very positive", "positive", "neutral", "negative", "very negative"]` |
| Aspect-based | `["product quality: positive", "shipping: negative", ...]` |
| Emotional | `["joy", "anger", "sadness", "fear", "surprise", "disgust"]` |

In [1]:
from gliclass import GLiClassModel, ZeroShotClassificationPipeline
from transformers import AutoTokenizer
import torch, json, os

fixtures = os.path.normpath(os.path.join(os.getcwd(), "..", "fixtures", "input"))
with open(os.path.join(fixtures, "product_reviews.json")) as f:
    reviews = json.load(f)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = GLiClassModel.from_pretrained("knowledgator/gliclass-edge-v3.0")
tokenizer = AutoTokenizer.from_pretrained("knowledgator/gliclass-edge-v3.0", add_prefix_space=True)

# Single-label: each review has one dominant sentiment
pipeline = ZeroShotClassificationPipeline(
    model, tokenizer,
    classification_type='single-label',
    device=device
)
print(f"✓ Loaded {len(reviews)} reviews")

✓ Loaded 15 reviews


## 1. Basic Polarity Detection

In [2]:
sentiment_labels = ["positive", "negative", "neutral"]
texts = [r["text"] for r in reviews]

results = pipeline(texts, sentiment_labels, batch_size=8)

correct = 0
print("Sentiment classification results:\n")
for review, res in zip(reviews, results):
    predicted = res[0]['label']
    score = res[0]['score']
    expected = review['sentiment']
    match = "✓" if predicted == expected else "✗"
    if predicted == expected:
        correct += 1
    print(f"  {match} predicted={predicted:8} expected={expected:8} score={score:.2f}")
    print(f"    {review['text'][:70]}...")

accuracy = correct / len(reviews)
print(f"\nAccuracy: {accuracy:.0%} ({correct}/{len(reviews)})")

  0%|          | 0/2 [00:00<?, ?it/s]

 50%|█████     | 1/2 [00:02<00:02,  2.34s/it]

100%|██████████| 2/2 [00:02<00:00,  1.18s/it]

Sentiment classification results:

  ✓ predicted=positive expected=positive score=0.88
    Absolutely love this headphone! The sound quality is crystal clear, ba...
  ✓ predicted=negative expected=negative score=1.00
    Complete waste of money. The charging cable broke after two days, batt...
  ✗ predicted=negative expected=neutral  score=0.93
    It works fine for the price. Nothing exceptional — the screen is decen...
  ✓ predicted=positive expected=positive score=0.86
    This blender completely transformed my morning routine. Smoothies come...
  ✓ predicted=negative expected=negative score=0.84
    The worst vacuum cleaner I have ever owned. Loses suction within minut...
  ✓ predicted=neutral  expected=neutral  score=0.98
    Arrived on time and matches the photos. The material is acceptable for...
  ✓ predicted=positive expected=positive score=0.95
    Outstanding camera! Photos in low light are stunning, the autofocus is...
  ✓ predicted=negative expected=negative score=1.00
   

## 2. Fine-Grained Sentiment

In [3]:
fine_grained = ["very positive", "positive", "neutral", "negative", "very negative"]

# Pick a clearly positive and clearly negative review
strong_positive = reviews[0]["text"]  # headphones review
strong_negative = reviews[1]["text"]  # earbuds review

for text, label in [(strong_positive, "positive"), (strong_negative, "negative")]:
    results = pipeline(text, fine_grained)[0]
    predicted = results[0]['label']
    print(f"  [{label:8}] → {predicted:15} ({results[0]['score']:.2f})")
    print(f"  {text[:70]}...")
    print()

  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00,  3.19it/s]

100%|██████████| 1/1 [00:00<00:00,  3.19it/s]

  [positive] → very positive   (0.67)
  Absolutely love this headphone! The sound quality is crystal clear, ba...



  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00, 92.12it/s]

  [negative] → negative        (0.53)
  Complete waste of money. The charging cable broke after two days, batt...



## 3. Aspect-Based Sentiment

GLiClass can detect sentiment for specific product aspects by using
descriptive labels like `"camera quality: positive"`.

In [4]:
# Multi-label pipeline for aspect detection
pipeline_ml = ZeroShotClassificationPipeline(
    model, tokenizer,
    classification_type='multi-label',
    device=device
)

review_text = """The sound quality is incredible and the design looks premium,
but the battery only lasts 4 hours and the app crashes constantly."""

aspect_labels = [
    "sound quality: positive",
    "sound quality: negative",
    "design: positive",
    "design: negative",
    "battery life: positive",
    "battery life: negative",
    "software / app: positive",
    "software / app: negative",
]

results = pipeline_ml(review_text, aspect_labels, threshold=0.4)[0]
print(f"Review: {review_text.strip()}\n")
print("Detected aspects:")
for r in sorted(results, key=lambda x: x['score'], reverse=True):
    print(f"  {r['label']:35} {r['score']:.3f}")

  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00, 50.34it/s]

Review: The sound quality is incredible and the design looks premium,
but the battery only lasts 4 hours and the app crashes constantly.

Detected aspects:
  software / app: negative            0.995
  battery life: negative              0.992
  sound quality: negative             0.991
  design: negative                    0.974


## 4. Emotion Detection

In [5]:
emotion_labels = ["joy", "anger", "sadness", "fear", "disgust", "surprise", "neutral"]

emotional_texts = [
    "I can't believe how amazing this is! This is the best day of my life!",
    "This is absolutely outrageous. I am furious about this terrible service.",
    "I'm really heartbroken. It just doesn't work anymore and I don't know why.",
]

results = pipeline(emotional_texts, emotion_labels, batch_size=4)
for text, res in zip(emotional_texts, results):
    emotion = res[0]['label']
    score = res[0]['score']
    print(f"  [{emotion:8}] ({score:.2f}) {text[:60]}...")

  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00, 76.09it/s]

  [joy     ] (0.87) I can't believe how amazing this is! This is the best day of...
  [anger   ] (0.53) This is absolutely outrageous. I am furious about this terri...
  [sadness ] (0.57) I'm really heartbroken. It just doesn't work anymore and I d...


## 5. Accuracy Evaluation

Measuring classification accuracy on a labeled dataset.

In [6]:
from collections import defaultdict

texts = [r["text"] for r in reviews]
results = pipeline(texts, ["positive", "negative", "neutral"], batch_size=8)
predictions = [res[0]['label'] for res in results]
ground_truth = [r['sentiment'] for r in reviews]

# Per-class accuracy
class_correct = defaultdict(int)
class_total = defaultdict(int)
for pred, truth in zip(predictions, ground_truth):
    class_total[truth] += 1
    if pred == truth:
        class_correct[truth] += 1

print("Per-class accuracy:")
for cls in ["positive", "negative", "neutral"]:
    acc = class_correct[cls] / class_total[cls] if class_total[cls] else 0
    print(f"  {cls:10} {acc:.0%} ({class_correct[cls]}/{class_total[cls]})")

overall = sum(class_correct.values()) / len(reviews)
print(f"\nOverall: {overall:.0%} ({sum(class_correct.values())}/{len(reviews)})")

  0%|          | 0/2 [00:00<?, ?it/s]

100%|██████████| 2/2 [00:00<00:00, 80.41it/s]

Per-class accuracy:
  positive   100% (5/5)
  negative   100% (5/5)
  neutral    40% (2/5)

Overall: 80% (12/15)


## Summary

- Use `'single-label'` for sentiment (positive / negative / neutral)
- Fine-grained labels (`"very positive"`, `"very negative"`) improve signal resolution
- Aspect-based sentiment: use `"aspect: sentiment"` label format with `'multi-label'`
- The model handles emotion detection zero-shot (joy, anger, sadness, etc.)
- GLiClass edge achieves ~82-85% accuracy on binary polarity tasks